# 🔤 Brahmi Script OCR — TrOCR Training on Google Colab

This notebook trains TrOCR on **combined Brahmi datasets** (synthetic + Capstone handwritten + BrahmiGAN + Kaggle) using Colab's **free GPU**.

**Pipeline:**
1. Mount Google Drive & clone project
2. Install dependencies
3. Generate synthetic dataset (~20K images)
4. Upload Extra datasets from Drive (Capstone, BrahmiGAN, Kaggle Archive)
5. Fine-tune `microsoft/trocr-small-printed` on combined ~107K images
6. Test inference
7. Save model to Drive

---

> ⚠️ **Before running:** Go to **Runtime → Change runtime type → GPU (T4)**

## Step 0 — Verify GPU is available

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU device      : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No GPU detected! Go to Runtime → Change runtime type → GPU")

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Clone project from GitHub

In [ ]:
import os

PROJECT_DIR = '/content/brahmi_ocr_project'

# Clone fresh from GitHub
if os.path.exists(PROJECT_DIR):
    !rm -rf {PROJECT_DIR}

!git clone https://github.com/tejaskarade100/brahmi_ocr_project.git {PROJECT_DIR}

%cd {PROJECT_DIR}
!ls

## Step 3 — Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## Step 4a — Generate synthetic Brahmi dataset

Generates **20,000 images** (5k chars + 10k words + 5k phrases) with augmentations.

Takes ~5–10 minutes.

In [ ]:
# Check if dataset already exists (skip if re-running)
labels_path = os.path.join(PROJECT_DIR, 'dataset', 'labels.txt')
if os.path.exists(labels_path):
    num_existing = sum(1 for line in open(labels_path) if line.strip())
    print(f"Dataset already exists with {num_existing} samples.")
    if num_existing >= 20000:
        print("Skipping generation. Delete dataset/labels.txt to regenerate.")
    else:
        print("Regenerating full dataset...")
        !python dataset/generate_synthetic.py --num_chars 5000 --num_words 10000 --num_phrases 5000
else:
    !python dataset/generate_synthetic.py --num_chars 5000 --num_words 10000 --num_phrases 5000

In [ ]:
# Verify dataset
!head -5 dataset/labels.txt
print(f"\nTotal images: {len(os.listdir('dataset/images')) - 1}")  # minus .gitkeep
print(f"Train split : {sum(1 for line in open('dataset/splits/train.txt') if line.strip())}")
print(f"Val split   : {sum(1 for line in open('dataset/splits/val.txt') if line.strip())}")
print(f"Test split  : {sum(1 for line in open('dataset/splits/test.txt') if line.strip())}")

## Step 4b — Upload Capstone handwritten dataset from Drive

Upload the **Capstone_Brahmi_Inscriptions** folder to your Google Drive first,
then this cell copies the RecognizerDataset to Colab.

⚠️ If you don't have the Capstone dataset on Drive, skip this cell.

In [ ]:
# Copy Capstone dataset from Drive to Colab workspace
import shutil

# Try multiple possible Drive paths
CAPSTONE_CANDIDATES = [
    '/content/drive/MyDrive/Project/brahmi_ocr_project/Capstone_Brahmi_Inscriptions/OCR/OCR_Dataset/RecognizerDataset_150_210',
    '/content/drive/MyDrive/Capstone_Brahmi_Inscriptions/OCR/OCR_Dataset/RecognizerDataset_150_210',
    '/content/drive/MyDrive/RecognizerDataset_150_210',
]

CAPSTONE_LOCAL = '/content/brahmi_ocr_project/capstone_data'
CAPSTONE_FOUND = False

for path in CAPSTONE_CANDIDATES:
    if os.path.exists(path):
        if not os.path.exists(CAPSTONE_LOCAL):
            print(f'Copying from {path}...')
            shutil.copytree(path, CAPSTONE_LOCAL)
        num_folders = len([d for d in os.listdir(CAPSTONE_LOCAL) if os.path.isdir(os.path.join(CAPSTONE_LOCAL, d))])
        print(f'✅ Capstone dataset ready: {num_folders} character folders')
        CAPSTONE_FOUND = True
        break

if not CAPSTONE_FOUND:
    print('⚠️ Capstone dataset not found on Drive.')


## Step 4c — Upload BrahmiGAN dataset from Drive

Upload the **BrahmiGAN** folder to your Google Drive first.

⚠️ If you don't have the BrahmiGAN dataset on Drive, skip this cell.

In [ ]:
# Copy BrahmiGAN dataset from Drive to Colab workspace
BRAHMIGAN_CANDIDATES = [
    '/content/drive/MyDrive/Project/brahmi_ocr_project/BrahmiGAN/BrahmiGAN',
    '/content/drive/MyDrive/BrahmiGAN/BrahmiGAN',
    '/content/drive/MyDrive/BrahmiGAN',
]

BRAHMIGAN_LOCAL = '/content/brahmi_ocr_project/brahmigan_data'
BRAHMIGAN_FOUND = False

for path in BRAHMIGAN_CANDIDATES:
    if os.path.exists(path):
        if not os.path.exists(BRAHMIGAN_LOCAL):
            print(f'Copying from {path}...')
            shutil.copytree(path, BRAHMIGAN_LOCAL)
        num_folders = len([d for d in os.listdir(BRAHMIGAN_LOCAL) if os.path.isdir(os.path.join(BRAHMIGAN_LOCAL, d))])
        print(f'✅ BrahmiGAN dataset ready: {num_folders} character folders')
        BRAHMIGAN_FOUND = True
        break

if not BRAHMIGAN_FOUND:
    print('⚠️ BrahmiGAN dataset not found on Drive. Skipping.')


## Step 4d — Upload Kaggle Archive dataset from Drive

Upload the **archive/train** folder to your Google Drive first.

⚠️ If you don't have the Kaggle dataset on Drive, skip this cell.

In [ ]:
# Copy Kaggle dataset from Drive to Colab workspace
KAGGLE_CANDIDATES = [
    '/content/drive/MyDrive/Project/brahmi_ocr_project/archive/train',
    '/content/drive/MyDrive/archive/train',
]

KAGGLE_LOCAL = '/content/brahmi_ocr_project/kaggle_data'
KAGGLE_FOUND = False

for path in KAGGLE_CANDIDATES:
    if os.path.exists(path):
        if not os.path.exists(KAGGLE_LOCAL):
            print(f'Copying from {path}...')
            shutil.copytree(path, KAGGLE_LOCAL)
        num_folders = len([d for d in os.listdir(KAGGLE_LOCAL) if os.path.isdir(os.path.join(KAGGLE_LOCAL, d))])
        print(f'✅ Kaggle dataset ready: {num_folders} character folders')
        KAGGLE_FOUND = True
        break

if not KAGGLE_FOUND:
    print('⚠️ Kaggle dataset not found on Drive. Skipping.')


## Step 5 — Train the TrOCR model 🚀

| Setting | Value |
|---------|-------|
| Base model | `microsoft/trocr-small-printed` |
| Epochs | 15 |
| Batch size | 8 |
| Learning rate | 3e-5 |
| FP16 | Auto (enabled on GPU) |
| Data | Synthetic (~20K) + Capstone (~60K) + BrahmiGAN (~21K) + Kaggle (~6K) |

**Estimated time:** ~60–120 min on T4 GPU

In [ ]:
# Build training command
EXTRA_DATA_PATHS = []
if CAPSTONE_FOUND:
    EXTRA_DATA_PATHS.append(CAPSTONE_LOCAL)
if BRAHMIGAN_FOUND:
    EXTRA_DATA_PATHS.append(BRAHMIGAN_LOCAL)
if KAGGLE_FOUND:
    EXTRA_DATA_PATHS.append(KAGGLE_LOCAL)

EXTRA_DATA_FLAG = ''
if len(EXTRA_DATA_PATHS) > 0:
    EXTRA_DATA_FLAG = f'--extra_data ' + ' '.join(EXTRA_DATA_PATHS)
    print(f'✅ Training with COMBINED data (synthetic + extra datasets)')
else:
    print(f'ℹ️ Training with SYNTHETIC data only')

!python training/train.py \
    --epochs 15 \
    --batch_size 8 \
    --lr 3e-5 \
    --data_dir dataset \
    --output_dir model/brahmi_trocr \
    {EXTRA_DATA_FLAG}

## Step 6 — Test inference on sample images

In [ ]:
# Load labels for ground truth comparison
labels = {}
for line in open('dataset/labels.txt', encoding='utf-8'):
    parts = line.strip().split('\t')
    if len(parts) == 2:
        labels[parts[0]] = parts[1]

# Run inference on first test image
test_images = [line.strip() for line in open('dataset/splits/test.txt') if line.strip()]
test_img = test_images[0]
print(f"Test image   : {test_img}")
print(f"Ground truth : {labels.get(test_img, 'N/A')}")
print()
!python inference/predict.py --image dataset/images/{test_img} --model_dir model/brahmi_trocr

In [ ]:
# Batch test: compare predictions vs ground truth
import sys
sys.path.insert(0, PROJECT_DIR)

from inference.predict import load_trained_model, predict

processor, model, device = load_trained_model('model/brahmi_trocr')
print(f"Model loaded on: {device}\n")

correct = 0
total = min(10, len(test_images))
for img_name in test_images[:total]:
    img_path = f'dataset/images/{img_name}'
    pred = predict(img_path, processor, model, device)
    gt = labels.get(img_name, '')
    match = '✅' if pred.strip() == gt.strip() else '❌'
    if pred.strip() == gt.strip():
        correct += 1
    print(f"{match} {img_name}  |  GT: {gt}  |  Pred: {pred}")

print(f"\nAccuracy: {correct}/{total} ({100*correct/total:.0f}%)")

## Step 7 — Save trained model to Google Drive

Copies the model to your Drive at: `My Drive/Project/brahmi_ocr_project/model/brahmi_trocr/`

In [ ]:
import shutil

# Save model to your Google Drive project folder
DRIVE_MODEL_DIR = '/content/drive/MyDrive/Project/brahmi_ocr_project/model/brahmi_trocr'

os.makedirs(os.path.dirname(DRIVE_MODEL_DIR), exist_ok=True)

if os.path.exists(DRIVE_MODEL_DIR):
    shutil.rmtree(DRIVE_MODEL_DIR)

shutil.copytree('model/brahmi_trocr', DRIVE_MODEL_DIR)
print(f"✅ Model saved to Google Drive!")
print(f"   Path: {DRIVE_MODEL_DIR}")
print(f"   Files: {os.listdir(DRIVE_MODEL_DIR)}")

In [ ]:
# Also save the generated dataset to Drive (so you don't have to regenerate)
DRIVE_DATASET_DIR = '/content/drive/MyDrive/Project/brahmi_ocr_project/dataset'

# Copy labels and splits (not images — they're too large for Drive)
shutil.copy2('dataset/labels.txt', os.path.join(DRIVE_DATASET_DIR, 'labels.txt'))
os.makedirs(os.path.join(DRIVE_DATASET_DIR, 'splits'), exist_ok=True)
for split_file in ['train.txt', 'val.txt', 'test.txt']:
    shutil.copy2(f'dataset/splits/{split_file}', os.path.join(DRIVE_DATASET_DIR, 'splits', split_file))

print(f"✅ Labels and splits saved to: {DRIVE_DATASET_DIR}")

## Done! 🎉

### Your trained model is saved at:
📁 `Google Drive / Project / brahmi_ocr_project / model / brahmi_trocr /`

### To use locally:
1. Download the `brahmi_trocr` folder from Drive to your local `model/` folder
2. Run inference:
   ```bash
   python inference/predict.py --image <your_image.png> --model_dir model/brahmi_trocr
   ```